In [0]:
%pip install confluent-kafka
dbutils.library.restartPython()

In [0]:
import importlib, kafka_conn
importlib.reload(kafka_conn)
from kafka_conn import get_kafka_conf, TOPIC
conf = get_kafka_conf(dbutils)
print("Config set. Topic:", TOPIC)

In [0]:
conf

In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StringType, LongType

schema = (StructType()
    .add("post_id", StringType())
    .add("type",    StringType())
    .add("user_id", StringType())
    .add("ts",      LongType()))

raw = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", conf["bootstrap.servers"])
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism",   "PLAIN")
    .option("kafka.sasl.jaas.config",
            f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '   # <-- kafkashaded. prefix
            f'username="{conf["sasl.username"]}" password="{conf["sasl.password"]}";')
    .option("subscribe", "post-engagement")
    .option("startingOffsets", "earliest")
    .load())

parsed = (raw
    .select(
        from_json(col("value").cast("string"), schema).alias("e"),
        col("partition"), col("offset"),
        col("timestamp").alias("kafka_ts"),
    )
    .select("e.post_id","e.type","e.user_id","e.ts",
            "partition","offset","kafka_ts")
    .withColumn("ingest_ts", current_timestamp()))

CHECKPOINT = "s3://retail-data-lake-dev/kafka/_checkpoint/post_engagement_raw"

query = (parsed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)          # <-- process all available data, then stop
    # .trigger(processingTime="5 seconds")
    .toTable("kafka_poc.engagement.post_engagement_raw"))

query.awaitTermination() 
query.stop()

In [0]:
raw = (spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", conf["bootstrap.servers"])
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism",   "PLAIN")
    .option("kafka.sasl.jaas.config",
            f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '   # <-- kafkashaded. prefix
            f'username="{conf["sasl.username"]}" password="{conf["sasl.password"]}";')
    .option("subscribe", "post-engagement")
    .option("startingOffsets", "earliest")
    .load())

In [0]:
raw_batch = (spark.read.format("kafka")          # read, NOT readStream
    .option("kafka.bootstrap.servers", conf["bootstrap.servers"])
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism",   "PLAIN")
    .option("kafka.sasl.jaas.config",
            f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{conf["sasl.username"]}" password="{conf["sasl.password"]}";')
    .option("subscribe", "post-engagement")
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load())

# value/key are binary — cast to string so they're readable
# display(
#     raw_batch.selectExpr(
#         "CAST(key AS STRING)   AS key",
#         "CAST(value AS STRING) AS value",
#         "partition",
#         "offset",
#         "timestamp"   
#     )
# )
raw_batch.display()

In [0]:
%fs
ls  s3://retail-data-lake-dev/kafka/_checkpoint/post_engagement_raw/offsets/

In [0]:
%fs
head s3://retail-data-lake-dev/kafka/_checkpoint/post_engagement_raw/offsets/0

###  Selecting the data

In [0]:
%sql
desc history kafka_poc.engagement.post_engagement_raw